# 📰 Notebook 2 — ChromaDB with News Articles

> **Goal:** the simplest end-to-end vector DB experience — Chroma auto-embeds for you.

## 🎯 Learning Goals
- Create a Chroma **collection** with built-in embedding
- Insert documents with metadata in one call
- Run **semantic + metadata-filtered** queries (`where` clause)
- Persist to disk and reload

## 🍱 Analogy
**Chroma = SQLite for vectors.**
- One file on disk. Zero servers. Zero config.
- It bundles the encoder, the index, the metadata store, and the documents.
- Perfect for prototypes, RAG demos, and laptops.

In [ ]:
# 📦 Install
# %pip install -q chromadb sentence-transformers pandas

In [ ]:
from IPython.display import SVG
SVG("""<svg viewBox="0 0 720 230" xmlns="http://www.w3.org/2000/svg" width="720"><style>text{font-family:Inter,system-ui,sans-serif}</style><rect width="720" height="230" fill="#f8fafc" rx="12"/><rect x="20" y="60" width="140" height="80" rx="10" fill="#dbeafe" stroke="#2563eb"/><text x="90" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">News</text><text x="90" y="110" text-anchor="middle" font-size="11" fill="#1e3a8a">20 articles</text><text x="90" y="125" text-anchor="middle" font-size="11" fill="#1e3a8a">+ category, date</text><rect x="200" y="60" width="140" height="80" rx="10" fill="#ede9fe" stroke="#7c3aed"/><text x="270" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">Auto-embed</text><text x="270" y="110" text-anchor="middle" font-size="11" fill="#5b21b6">default = MiniLM</text><text x="270" y="125" text-anchor="middle" font-size="11" fill="#5b21b6">no manual encode</text><rect x="380" y="60" width="140" height="80" rx="10" fill="#fef3c7" stroke="#f59e0b"/><text x="450" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#78350f">Collection</text><text x="450" y="110" text-anchor="middle" font-size="11" fill="#78350f">vectors + metadata</text><text x="450" y="125" text-anchor="middle" font-size="11" fill="#78350f">+ documents</text><rect x="560" y="60" width="140" height="80" rx="10" fill="#dcfce7" stroke="#15803d"/><text x="630" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#065f46">Query + Filter</text><text x="630" y="110" text-anchor="middle" font-size="11" fill="#065f46">where=category</text><text x="630" y="125" text-anchor="middle" font-size="11" fill="#065f46">+ semantic</text><text x="172" y="105" font-size="22" fill="#475569">→</text><text x="352" y="105" font-size="22" fill="#475569">→</text><text x="532" y="105" font-size="22" fill="#475569">→</text><text x="360" y="35" text-anchor="middle" font-size="14" font-weight="700" fill="#0f172a">ChromaDB pipeline</text><text x="360" y="180" text-anchor="middle" font-size="11" fill="#475569">Chroma stores vectors, metadata, and the original document together</text></svg>""")

In [ ]:
# 📰 News articles spanning Tech / Sports / Politics / Health / Business
articles = [
    {"id": "n01", "category": "Tech",     "date": "2024-03-12", "title": "OpenAI releases GPT-4 Turbo",       "text": "OpenAI announced an upgraded GPT-4 model with longer context windows and reduced API pricing for developers building chat applications."},
    {"id": "n02", "category": "Tech",     "date": "2024-04-02", "title": "Apple unveils new M3 chips",        "text": "Apple introduced its M3 family of processors built on a 3-nanometer process, promising major leaps in laptop performance and battery life."},
    {"id": "n03", "category": "Tech",     "date": "2024-05-20", "title": "Google's Gemini becomes multimodal","text": "Google's Gemini model now natively understands images and audio alongside text, marking a shift in how chatbots reason about content."},
    {"id": "n04", "category": "Tech",     "date": "2024-06-15", "title": "Meta open-sources Llama 3",        "text": "Meta released Llama 3 weights, allowing companies to run powerful language models on their own infrastructure without paying API fees."},
    {"id": "n05", "category": "Sports",   "date": "2024-07-04", "title": "Olympic torch arrives in Paris",   "text": "The Olympic torch reached Paris ahead of the Summer Games, kicking off two weeks of athletic competition across dozens of disciplines."},
    {"id": "n06", "category": "Sports",   "date": "2024-08-11", "title": "Manchester City wins Premier League","text": "Manchester City clinched another Premier League title with a dramatic final-day victory, extending their dominance in English football."},
    {"id": "n07", "category": "Sports",   "date": "2024-09-09", "title": "US Open tennis final upset",       "text": "An unseeded young player defeated the world number one in five sets to claim a surprise US Open tennis championship."},
    {"id": "n08", "category": "Politics", "date": "2024-10-01", "title": "EU passes new AI regulation",      "text": "The European Union finalized its AI Act, introducing strict rules on high-risk AI systems and fines for companies violating transparency standards."},
    {"id": "n09", "category": "Politics", "date": "2024-10-22", "title": "UK announces snap election",       "text": "The British Prime Minister called a surprise general election, citing the need for a fresh mandate to address economic and security challenges."},
    {"id": "n10", "category": "Politics", "date": "2024-11-05", "title": "US presidential election results", "text": "American voters went to the polls in a closely watched presidential election that hinged on results in a handful of swing states."},
    {"id": "n11", "category": "Health",   "date": "2024-02-14", "title": "New Alzheimer's drug approved",    "text": "Regulators approved a new monoclonal antibody therapy shown to slow cognitive decline in early-stage Alzheimer's patients."},
    {"id": "n12", "category": "Health",   "date": "2024-03-30", "title": "WHO warns of new flu strain",      "text": "The World Health Organization is monitoring a novel influenza strain detected in livestock, urging early surveillance and vaccine readiness."},
    {"id": "n13", "category": "Health",   "date": "2024-05-08", "title": "Gene therapy for sickle cell",     "text": "A breakthrough gene-editing therapy has cured sickle cell disease in a clinical trial, offering hope to millions of patients worldwide."},
    {"id": "n14", "category": "Business", "date": "2024-04-18", "title": "Tesla cuts car prices globally",   "text": "Tesla reduced prices across all its electric vehicle models in a bid to defend market share from growing Chinese EV competition."},
    {"id": "n15", "category": "Business", "date": "2024-06-25", "title": "Nvidia hits $3 trillion valuation","text": "Nvidia briefly became the world's most valuable company as demand for AI accelerator chips continued to surge across cloud providers."},
    {"id": "n16", "category": "Business", "date": "2024-09-20", "title": "Federal Reserve cuts interest rates","text": "The US Federal Reserve cut interest rates for the first time in four years, citing easing inflation and signs of a softening labor market."},
    {"id": "n17", "category": "Tech",     "date": "2024-07-30", "title": "Microsoft expands Copilot to Excel","text": "Microsoft rolled out Copilot AI features deeply into Excel, allowing users to write formulas, analyze sheets, and generate charts via chat."},
    {"id": "n18", "category": "Sports",   "date": "2024-12-01", "title": "F1 season finale in Abu Dhabi",    "text": "The Formula One world championship was decided at the Abu Dhabi Grand Prix in a tense final-lap battle between two title contenders."},
    {"id": "n19", "category": "Health",   "date": "2024-08-22", "title": "Wearables detect heart problems",  "text": "A new study found that consumer smartwatches can reliably detect atrial fibrillation, prompting earlier diagnosis and treatment."},
    {"id": "n20", "category": "Business", "date": "2024-11-12", "title": "Amazon expands grocery delivery",  "text": "Amazon announced expanded same-day grocery delivery in over a hundred US cities, intensifying competition with Walmart and Instacart."},
]
import pandas as pd
pd.DataFrame(articles).head()

In [ ]:
# 🚀 Create an in-memory Chroma client and a collection
import chromadb
from chromadb.utils import embedding_functions

# Built-in embedding function — uses sentence-transformers under the hood
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client = chromadb.Client()                      # in-memory; use PersistentClient(path=...) for disk
collection = client.create_collection(
    name="news",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},          # distance metric
)
print("Collection ready:", collection.name)

In [ ]:
# 📥 Insert all articles in ONE call — Chroma embeds them automatically
collection.add(
    ids=[a["id"] for a in articles],
    documents=[a["text"] for a in articles],
    metadatas=[{"title": a["title"], "category": a["category"], "date": a["date"]} for a in articles],
)
print("Total docs in collection:", collection.count())

In [ ]:
# 🔍 Semantic search — no embedding step on our side
def show(results, query):
    print(f"\n🔎 \"{query}\"")
    rows = []
    for i in range(len(results["ids"][0])):
        rows.append({
            "title": results["metadatas"][0][i]["title"],
            "category": results["metadatas"][0][i]["category"],
            "distance": round(results["distances"][0][i], 3),
        })
    return pd.DataFrame(rows)

q = "artificial intelligence and large language models"
show(collection.query(query_texts=[q], n_results=5), q)

In [ ]:
# 🎯 Filtered query — semantic search WITHIN a category (where clause)
q = "championship victory"
res = collection.query(query_texts=[q], n_results=5, where={"category": "Sports"})
show(res, q + " [Sports only]")

In [ ]:
# 🧮 Combined filters — multiple metadata conditions
res = collection.query(
    query_texts=["regulation and government policy"],
    n_results=5,
    where={"$and": [{"category": "Politics"}, {"date": {"$gte": "2024-10-01"}}]},
)
show(res, "Politics articles after Oct 2024")

In [ ]:
# 💾 Persist to disk so you can reload later
persistent = chromadb.PersistentClient(path="./chroma_news_db")
col2 = persistent.get_or_create_collection(name="news", embedding_function=embed_fn)
col2.add(
    ids=[a["id"] for a in articles],
    documents=[a["text"] for a in articles],
    metadatas=[{"title": a["title"], "category": a["category"], "date": a["date"]} for a in articles],
)
print("Persisted collection size:", col2.count(), "→ folder ./chroma_news_db")

## 🏋️ Exercises
1. Add 3 **Entertainment** articles. Query "movies and streaming wars" and confirm filtering by `category="Entertainment"` works.
2. Use `collection.update(...)` to change the title of one article. Query again to see metadata refresh.
3. Switch the embedding function to `OpenAIEmbeddingFunction` (requires API key) — does ranking quality improve?

## 🎓 Recap — When to use ChromaDB
✅ RAG prototype on a laptop · zero infra · auto-embedding
✅ Documents + vectors + metadata in one place
❌ Need 100M+ vectors, multi-tenant, distributed → use Qdrant / Milvus / Pinecone